In [7]:
import pandas as pd
import numpy as np

In [8]:
# load the dataset
series = pd.read_csv('daily-total-female-births.csv', header=0, index_col=0)
series

,Births
Date,
1959-01-01,35
1959-01-02,32
1959-01-03,30
1959-01-04,31
1959-01-05,44
...,...
1959-12-27,37
1959-12-28,52
1959-12-29,48


In [9]:
# One-Step Univariate Forecasting

def series_to_supervised(data, n_in = 1, n_out = 1, dropnan = True):
    n_vars = 1 if type(data) is list else data.shape[1]
    df = pd.DataFrame(data)
    cols, names = list(), list()

    # input sequence (t-n, ..., t - 1)
    for i in range(n_in, 0, -1):
        cols.append(df.shift(i))
        names += [('var%d(t-%d)' % (j + 1, i)) for j in range(n_vars)]

    # forecast sequence (t, t + 1, ..., t + n)
    for i in range(0, n_out):
        cols.append(df.shift(-i))
        if i == 0:
            names += [('var%d(t)' % (j + 1)) for j in range(n_vars)]
        else:
            names += [('var%d(t + %d)' % (j + 1, i)) for j in range(n_vars)]

    # put it all together
    agg = pd.concat(cols, axis = 1)
    agg.columns = names

    # drop rows with NaN values
    if dropnan:
        agg.dropna(inplace = True)
    return agg

In [23]:
values = series.values
# tranform the time series data into supervised learning
train = series_to_supervised(values, n_in=6).values
print(train)

[[35. 32. 30. ... 44. 29. 45.]
 [32. 30. 31. ... 29. 45. 43.]
 [30. 31. 44. ... 45. 43. 38.]
 ...
 [40. 38. 44. ... 37. 52. 48.]
 [38. 44. 34. ... 52. 48. 55.]
 [44. 34. 37. ... 48. 55. 50.]]


In [21]:
from sklearn.ensemble import RandomForestRegressor
# split into input and output columns
trainX, trainy = train[:, :-1], train[:, -1]

# fit model
model = RandomForestRegressor(n_estimators=1000)
model.fit(trainX, trainy)

# construct an input for a new prediction
row = values[-6:].flatten()

# make a one-step prediction
yhat = model.predict(np.asarray([row]))
print('Input: %s, Predicted: %.3f' % (row, yhat[0]))


Input: [34 37 52 48 55 50], Predicted: 43.074
